In [19]:
import time
import math
import cupy as cp
import numpy as np
delimiter = '\t'
newline = '\n'

file = 'Transactional_BMS_POS.csv'

In [20]:
start = time.time()

with open(file, 'rb') as f:
    data = f.read()
    byte_array = cp.frombuffer(data, dtype=cp.uint8)

line_pos = cp.where(byte_array == ord(newline))[0]
line_pos = cp.array([0] + line_pos.tolist() + [byte_array.shape[0]])
line_count = line_pos.shape[0] - 1
line_lengths = line_pos[1:] - line_pos[:-1]

rk = cp.RawKernel(r"""
extern "C" __global__
void count_items_per_line(const unsigned char* byte_array,
                          const long* line_pos,
                          const long line_count,
                          const unsigned char delimiter,
                          long* items_per_line) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < line_count) {
        long start = line_pos[idx];
        long end = line_pos[idx + 1];
        long count = 0;
        for (long i = start; i < end; ++i) {
            if (byte_array[i] == delimiter) {
                count++;
            }
        }
        items_per_line[idx] = count + 1; // +1 for last item
    }   
}
""", 'count_items_per_line') 

items_per_line = cp.zeros(line_count, dtype=cp.int64)
threads_per_block = 256
blocks_per_grid = (line_count + (threads_per_block - 1)) // threads_per_block   
rk((blocks_per_grid,), (threads_per_block,),
   (byte_array, line_pos, line_count, ord(delimiter), items_per_line))

total_items = cp.sum(items_per_line).item()
items_array = cp.zeros(total_items, dtype=cp.int64)
index_array = cp.append(cp.zeros(1, dtype=cp.int64), items_per_line)
index_array = cp.cumsum(index_array)

rk_parse = cp.RawKernel(r"""
extern "C" __device__ 
long parse_number(const unsigned char* byte_array, long start, long end) {
    long num = 0;
    for (long i = start; i < end; ++i) {
        num = num * 10 + (byte_array[i] - '0');
    }
    return num;
}
                        
extern "C" __global__
void parse_file(const unsigned char* byte_array,
                const long* line_pos,
                const long* index_array,
                const long line_count,
                const unsigned char delimiter,
                long* items_array) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < line_count) {     
        long start = line_pos[idx] + 1;
        if (idx == 0)
            start = line_pos[idx];
        long end = line_pos[idx + 1];
        long item_start = start;
        long item_idx = index_array[idx];
        for (long i = start; i < end; ++i) {
            if (byte_array[i] == delimiter || i == end - 2) {
                long item_end = (byte_array[i] == delimiter) ? i : i + 1;
                items_array[item_idx] = parse_number(byte_array, item_start, item_end);
                item_idx++;
                item_start = i + 1;
            }
        }
    }
}
""", 'parse_file')

threads_per_block = 256
blocks_per_grid = (line_count + (threads_per_block - 1)) // threads_per_block
rk_parse((blocks_per_grid,), (threads_per_block,),
         (byte_array, line_pos, index_array, line_count, ord(delimiter), items_array))  

end = time.time()
print("Time taken (s):", end - start)

print("Items Array:", items_array)
print("Index Array:", index_array)

Time taken (s): 0.37981462478637695
Items Array: [     1      0      1 ... 515552     92      0]
Index Array: [      0       2       4 ... 6733166 6733168 6733169]


# CUDA Kernel Codes

In [21]:
KERNEL_CODE = r"""
extern "C" __device__
int clz32(unsigned int x) { return (x == 0) ? 32 : __clz(x); }

extern "C" __device__
int clz64(unsigned long long x) { return (x == 0) ? 64 : __clzll(x); }

extern "C" __device__
unsigned int murmur3_32(long v) {
    unsigned int h = (unsigned int)v;
    h ^= h >> 16; h *= 0x85ebca6b;
    h ^= h >> 13; h *= 0xc2b2ae35;
    h ^= h >> 16; return h;
}

extern "C" __device__
unsigned long long murmur3_64(long v) {
    unsigned long long h = (unsigned long long)v;
    h ^= h >> 33; h *= 0xff51afd7ed558ccdULL;
    h ^= h >> 33; h *= 0xc4ceb9fe1a85ec53ULL;
    h ^= h >> 33; return h;
}

extern "C" __device__
unsigned int cityhash_32(long v) {
    unsigned int h = (unsigned int)v;
    unsigned int k = 0x9ae16a3b;
    

    h ^= k;
    h *= 0x85ebca6b;
    h ^= (h >> 13);
    h ^= k;
    h *= 0xc2b2ae35;
    h ^= (h >> 16);
    h *= 0x85ebca6b;
    
    return h;
}

extern "C" __device__
unsigned long long cityhash_64(long v) {
    unsigned long long h = (unsigned long long)v;
    unsigned long long k = 0x9ae16a3b2f90404fULL;
    
    h ^= k;
    h *= 0x9ddfea08eb382d69ULL;
    h ^= (h >> 47);
    h ^= k;
    h *= 0x9ddfea08eb382d69ULL;
    h ^= (h >> 47);
    h *= 0x9ddfea08eb382d69ULL;
    
    return h;
}

extern "C" __device__
unsigned int farmhash_32(long v) {
    unsigned int h = (unsigned int)v;
    

    const unsigned int mul = 0x85ebca6b;
    unsigned int a = h * mul;
    a ^= (a >> 16);
    unsigned int b = (h ^ a) * mul;
    b ^= (b >> 16);
    b *= mul;
    
    b ^= (b >> 13);
    b *= 0xc2b2ae35;
    b ^= (b >> 15);
    
    return b;
}

extern "C" __device__
unsigned long long farmhash_64(long v) {
    unsigned long long h = (unsigned long long)v;
    
    const unsigned long long mul = 0x9ddfea08eb382d69ULL;
    unsigned long long a = h * mul;
    a ^= (a >> 47);
    unsigned long long b = (h ^ a) * mul;
    b ^= (b >> 47);
    b *= mul;
    
    b ^= (b >> 44);
    b *= 0xc4ceb9fe1a85ec53ULL;
    b ^= (b >> 41);
    
    return b;
}

extern "C" __global__
void hll_murmur_32(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned int hash = murmur3_32(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz32(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_murmur_64(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned long long hash = murmur3_64(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz64(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_cityhash_32(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned int hash = cityhash_32(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz32(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_cityhash_64(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned long long hash = cityhash_64(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz64(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_farmhash_32(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned int hash = farmhash_32(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz32(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_farmhash_64(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned long long hash = farmhash_64(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz64(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}
"""

# Complile Kernels

In [22]:
kernels = {
    ('murmur', 32): cp.RawKernel(KERNEL_CODE, 'hll_murmur_32'),
    ('murmur', 64): cp.RawKernel(KERNEL_CODE, 'hll_murmur_64'),
    ('cityhash', 32): cp.RawKernel(KERNEL_CODE, 'hll_cityhash_32'),
    ('cityhash', 64): cp.RawKernel(KERNEL_CODE, 'hll_cityhash_64'),
    ('farmhash', 32): cp.RawKernel(KERNEL_CODE, 'hll_farmhash_32'),
    ('farmhash', 64): cp.RawKernel(KERNEL_CODE, 'hll_farmhash_64'),
}

In [23]:
def run_hll(items, b, hash_type, bits, true_card, runs=10):
    m = 1 << b
    alpha = 0.7213 / (1 + 1.078 / m) if m >= 128 else 0.5
    regs = cp.zeros(m, dtype=cp.int32)
    kernel = kernels[(hash_type, bits)]
    
    #warm up
    blocks = (len(items) + 255) // 256
    kernel((blocks,), (256,), (items, len(items), b, regs))
    cp.cuda.Stream.null.synchronize()
    
    regs.fill(0)
    times = []
    for _ in range(runs):
        start = time.time()
        kernel((blocks,), (256,), (items, len(items), b, regs))
        cp.cuda.Stream.null.synchronize()
        times.append(time.time() - start)
    
    # Estimate cardinality
    raw_est = alpha * m * m / cp.sum(cp.power(2.0, -regs.astype(cp.float64)))
    zeros = cp.sum(regs == 0)
    if raw_est <= 2.5 * m and zeros != 0:
        estimate = m * cp.log(m / zeros)
    else:
        estimate = raw_est
    
    error = abs(float(estimate) - true_card) / true_card * 100
    return np.mean(times) * 1000, error, float(estimate)

In [24]:
true_card = len(cp.unique(items_array))
print(f"Dataset: {len(items_array):,} items")
print(f"True cardinality: {true_card:,}")

Dataset: 6,733,169 items
True cardinality: 515,553


In [25]:
print("EXPERIMENT A: HASH FUNCTION IMPACT")
print("Fixed: 1024 buckets (b=10)")
print("="*70)

exp_a_results = []

for hash_type in ['murmur', 'cityhash', 'farmhash']:
    for bits in [32, 64]:
        runtime, error, est = run_hll(items_array, 10, hash_type, bits, true_card)
        exp_a_results.append((hash_type, bits, runtime, error))
        print(f"{hash_type:9} {bits}-bit: {runtime:7.3f} ms  |  Error: {error:6.3f}%")

best_hash, best_bits = min(exp_a_results, key=lambda x: (x[3], x[2]))[:2]
print(f"\nBest hash function: {best_hash}-{best_bits}")

EXPERIMENT A: HASH FUNCTION IMPACT
Fixed: 1024 buckets (b=10)
murmur    32-bit:   0.918 ms  |  Error:  2.039%
murmur    64-bit:   0.692 ms  |  Error:  0.850%
cityhash  32-bit:   0.734 ms  |  Error:  0.913%
cityhash  64-bit:   0.731 ms  |  Error:  0.758%
farmhash  32-bit:   0.723 ms  |  Error:  0.417%
farmhash  64-bit:   0.742 ms  |  Error:  0.885%

Best hash function: farmhash-32


In [26]:
print("EXPERIMENT B: BUCKET COUNT IMPACT")
print(f"Fixed: {best_hash}-{best_bits} hash function")
print("="*70)

exp_b_results = []

for b in [6, 8, 10, 12, 14, 16]:
    runtime, error, est = run_hll(items_array, b, best_hash, best_bits, true_card)
    mem_kb = (1 << b) * 4 / 1024
    exp_b_results.append((b, runtime, error, mem_kb))
    print(f"b={b:2} ({1<<b:5} buckets): {runtime:7.3f} ms  |  "
          f"Error: {error:6.3f}%  |  Mem: {mem_kb:6.2f} KB")

EXPERIMENT B: BUCKET COUNT IMPACT
Fixed: farmhash-32 hash function
b= 6 (   64 buckets):   3.207 ms  |  Error: 10.996%  |  Mem:   0.25 KB
b= 8 (  256 buckets):   2.288 ms  |  Error:  9.744%  |  Mem:   1.00 KB
b=10 ( 1024 buckets):   0.711 ms  |  Error:  0.417%  |  Mem:   4.00 KB
b=12 ( 4096 buckets):   0.299 ms  |  Error:  1.809%  |  Mem:  16.00 KB
b=14 (16384 buckets):   0.328 ms  |  Error:  0.984%  |  Mem:  64.00 KB
b=16 (65536 buckets):   0.283 ms  |  Error:  0.477%  |  Mem: 256.00 KB


In [27]:
best_a = min(exp_a_results, key=lambda x: x[3])  # Most accurate
fastest_a = min(exp_a_results, key=lambda x: x[2])  # Fastest
best_b = min(exp_b_results, key=lambda x: x[2])
fastest_b = min(exp_b_results, key=lambda x: x[1])


print("RECOMMENDED CONFIGURATION")
print(f"{'='*70}")
print(f"Hash Function: {best_hash}-{best_bits}")
print(f"Bucket Count: {1<<best_b[0]} (b={best_b[0]})")
print(f"Expected Error: {best_b[2]:.3f}%")
print(f"Expected Runtime: {best_b[1]:.3f} ms")
print(f"Memory Usage: {best_b[3]:.2f} KB")

RECOMMENDED CONFIGURATION
Hash Function: farmhash-32
Bucket Count: 1024 (b=10)
Expected Error: 0.417%
Expected Runtime: 0.711 ms
Memory Usage: 4.00 KB


In [1]:
import time
import math
import cupy as cp
import numpy as np
delimiter = '\t'
newline = '\n'

file = 'Transactional_retail.csv'

In [2]:
start = time.time()

with open(file, 'rb') as f:
    data = f.read()
    byte_array = cp.frombuffer(data, dtype=cp.uint8)

line_pos = cp.where(byte_array == ord(newline))[0]
line_pos = cp.array([0] + line_pos.tolist() + [byte_array.shape[0]])
line_count = line_pos.shape[0] - 1

rk = cp.RawKernel(r"""
extern "C" __global__
void count_items_per_line(const unsigned char* byte_array,
                          const long* line_pos,
                          const long line_count,
                          const unsigned char delimiter,
                          long* items_per_line) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < line_count) {
        long start = line_pos[idx];
        long end = line_pos[idx + 1];
        long count = 0;
        for (long i = start; i < end; ++i) {
            if (byte_array[i] == delimiter) {
                count++;
            }
        }
        items_per_line[idx] = count + 1;
    }   
}
""", 'count_items_per_line')

items_per_line = cp.zeros(line_count, dtype=cp.int64)
threads_per_block = 256
blocks_per_grid = (line_count + (threads_per_block - 1)) // threads_per_block
rk((blocks_per_grid,), (threads_per_block,),
   (byte_array, line_pos, line_count, ord(delimiter), items_per_line))

total_items = int(cp.sum(items_per_line).item())
items_array = cp.zeros(total_items, dtype=cp.int64)
index_array = cp.append(cp.zeros(1, dtype=cp.int64), items_per_line)
index_array = cp.cumsum(index_array)

rk_parse = cp.RawKernel(r"""
extern "C" __device__ 
long parse_number(const unsigned char* byte_array, long start, long end) {
    long num = 0;
    for (long i = start; i < end; ++i) {
        num = num * 10 + (byte_array[i] - '0');
    }
    return num;
}
                        
extern "C" __global__
void parse_file(const unsigned char* byte_array,
                const long* line_pos,
                const long* index_array,
                const long line_count,
                const unsigned char delimiter,
                long* items_array) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < line_count) {     
        long start = line_pos[idx] + 1;
        if (idx == 0)
            start = line_pos[idx];
        long end = line_pos[idx + 1];
        long item_start = start;
        long item_idx = index_array[idx];
        for (long i = start; i < end; ++i) {
            if (byte_array[i] == delimiter || i == end - 2) {
                long item_end = (byte_array[i] == delimiter) ? i : i + 1;
                items_array[item_idx] = parse_number(byte_array, item_start, item_end);
                item_idx++;
                item_start = i + 1;
            }
        }
    }
}
""", 'parse_file')

threads_per_block = 256
blocks_per_grid = (line_count + (threads_per_block - 1)) // threads_per_block
rk_parse((blocks_per_grid,), (threads_per_block,),
         (byte_array, line_pos, index_array, line_count, ord(delimiter), items_array))

In [3]:
KERNEL_CODE = r"""
extern "C" __device__
int clz32(unsigned int x) { return (x == 0) ? 32 : __clz(x); }

extern "C" __device__
int clz64(unsigned long long x) { return (x == 0) ? 64 : __clzll(x); }

extern "C" __device__
unsigned int murmur3_32(long v) {
    unsigned int h = (unsigned int)v;
    h ^= h >> 16; h *= 0x85ebca6b;
    h ^= h >> 13; h *= 0xc2b2ae35;
    h ^= h >> 16; return h;
}

extern "C" __device__
unsigned long long murmur3_64(long v) {
    unsigned long long h = (unsigned long long)v;
    h ^= h >> 33; h *= 0xff51afd7ed558ccdULL;
    h ^= h >> 33; h *= 0xc4ceb9fe1a85ec53ULL;
    h ^= h >> 33; return h;
}

extern "C" __device__
unsigned int cityhash_32(long v) {
    unsigned int h = (unsigned int)v;
    unsigned int k = 0x9ae16a3b;
    h ^= k;
    h *= 0x85ebca6b;
    h ^= (h >> 13);
    h ^= k;
    h *= 0xc2b2ae35;
    h ^= (h >> 16);
    h *= 0x85ebca6b;
    return h;
}

extern "C" __device__
unsigned long long cityhash_64(long v) {
    unsigned long long h = (unsigned long long)v;
    unsigned long long k = 0x9ae16a3b2f90404fULL;
    h ^= k;
    h *= 0x9ddfea08eb382d69ULL;
    h ^= (h >> 47);
    h ^= k;
    h *= 0x9ddfea08eb382d69ULL;
    h ^= (h >> 47);
    h *= 0x9ddfea08eb382d69ULL;
    return h;
}

extern "C" __device__
unsigned int farmhash_32(long v) {
    unsigned int h = (unsigned int)v;
    const unsigned int mul = 0x85ebca6b;
    unsigned int a = h * mul;
    a ^= (a >> 16);
    unsigned int b = (h ^ a) * mul;
    b ^= (b >> 16);
    b *= mul;
    b ^= (b >> 13);
    b *= 0xc2b2ae35;
    b ^= (b >> 15);
    return b;
}

extern "C" __device__
unsigned long long farmhash_64(long v) {
    unsigned long long h = (unsigned long long)v;
    const unsigned long long mul = 0x9ddfea08eb382d69ULL;
    unsigned long long a = h * mul;
    a ^= (a >> 47);
    unsigned long long b = (h ^ a) * mul;
    b ^= (b >> 47);
    b *= mul;
    b ^= (b >> 44);
    b *= 0xc4ceb9fe1a85ec53ULL;
    b ^= (b >> 41);
    return b;
}

extern "C" __global__
void hll_murmur_32(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned int hash = murmur3_32(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz32(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_murmur_64(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned long long hash = murmur3_64(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz64(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_cityhash_32(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned int hash = cityhash_32(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz32(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_cityhash_64(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned long long hash = cityhash_64(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz64(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_farmhash_32(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned int hash = farmhash_32(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz32(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}

extern "C" __global__
void hll_farmhash_64(const long* items, long n, int b, int* regs) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        unsigned long long hash = farmhash_64(items[idx]);
        int bucket = hash & ((1 << b) - 1);
        int lz = clz64(hash >> b) - b + 1;
        atomicMax(&regs[bucket], lz);
    }
}
"""

In [4]:
kernels = {
    ('murmur', 32): cp.RawKernel(KERNEL_CODE, 'hll_murmur_32'),
    ('murmur', 64): cp.RawKernel(KERNEL_CODE, 'hll_murmur_64'),
    ('cityhash', 32): cp.RawKernel(KERNEL_CODE, 'hll_cityhash_32'),
    ('cityhash', 64): cp.RawKernel(KERNEL_CODE, 'hll_cityhash_64'),
    ('farmhash', 32): cp.RawKernel(KERNEL_CODE, 'hll_farmhash_32'),
    ('farmhash', 64): cp.RawKernel(KERNEL_CODE, 'hll_farmhash_64'),
}

In [5]:
def run_hll(items, b, hash_type, bits, true_card, runs=10):
    m = 1 << b
    alpha = 0.7213 / (1 + 1.078 / m) if m >= 128 else 0.5
    regs = cp.zeros(m, dtype=cp.int32)
    kernel = kernels[(hash_type, bits)]
    blocks = (len(items) + 255) // 256
    kernel((blocks,), (256,), (items, len(items), b, regs))
    cp.cuda.Stream.null.synchronize()
    regs.fill(0)
    times = []
    for _ in range(runs):
        start = time.time()
        kernel((blocks,), (256,), (items, len(items), b, regs))
        cp.cuda.Stream.null.synchronize()
        times.append(time.time() - start)
    raw_est = alpha * m * m / cp.sum(cp.power(2.0, -regs.astype(cp.float64)))
    zeros = cp.sum(regs == 0)
    if raw_est <= 2.5 * m and zeros != 0:
        estimate = m * cp.log(m / zeros)
    else:
        estimate = raw_est
    error = abs(float(estimate) - true_card) / true_card * 100
    return np.mean(times) * 1000, error, float(estimate)

In [6]:
true_card = len(cp.unique(items_array))
dataset_size = len(items_array)
print(true_card, dataset_size)

16469 908173


In [7]:
print("EXPERIMENT A: HASH FUNCTION IMPACT")
print("1024 buckets (b=10)\n")
exp_a_results = []
for hash_type in ['murmur', 'cityhash', 'farmhash']:
    for bits in [32, 64]:
        runtime, error, est = run_hll(items_array, 10, hash_type, bits, true_card)
        exp_a_results.append((hash_type, bits, runtime, error, est))
        print(f"{hash_type:9} {bits}-bit: {runtime:7.3f} ms {error:6.3f}%")
best_hash, best_bits = min(exp_a_results, key=lambda x: (x[3], x[2]))[:2]
print(f"\nBest hash function: {best_hash}-{best_bits}")

EXPERIMENT A: HASH FUNCTION IMPACT
1024 buckets (b=10)

murmur    32-bit:   0.160 ms  0.537%
murmur    64-bit:   0.139 ms  3.249%
cityhash  32-bit:   0.167 ms  0.560%
cityhash  64-bit:   0.133 ms  5.712%
farmhash  32-bit:   0.143 ms  1.577%
farmhash  64-bit:   0.137 ms  4.073%

Best hash function: murmur-32


In [8]:
print("EXPERIMENT B: BUCKET COUNT IMPACT")
print(f"{best_hash}-{best_bits} hash function\n")
exp_b_results = []
for b in [6, 8, 10, 12, 14, 16]:
    runtime, error, est = run_hll(items_array, b, best_hash, best_bits, true_card)
    mem_kb = (1 << b) * 4 / 1024
    exp_b_results.append((b, runtime, error, mem_kb, est))
    print(f"b={b:2} ({1<<b:5} buckets): {runtime:7.3f} ms  {error:6.3f}%  {mem_kb:6.2f} KB")

EXPERIMENT B: BUCKET COUNT IMPACT
murmur-32 hash function

b= 6 (   64 buckets):   0.335 ms  39.029%    0.25 KB
b= 8 (  256 buckets):   0.314 ms   5.305%    1.00 KB
b=10 ( 1024 buckets):   0.166 ms   0.537%    4.00 KB
b=12 ( 4096 buckets):   0.079 ms   0.787%   16.00 KB
b=14 (16384 buckets):   0.062 ms   0.418%   64.00 KB
b=16 (65536 buckets):   0.065 ms   0.182%  256.00 KB


In [9]:
HASH_TABLE_KERNEL = r"""
extern "C" __device__
unsigned int farmhash_32(long v) {
    unsigned int h = (unsigned int)v;
    const unsigned int mul = 0x85ebca6b;
    unsigned int a = h * mul;
    a ^= (a >> 16);
    unsigned int b = (h ^ a) * mul;
    b ^= (b >> 16);
    b *= mul;
    b ^= (b >> 13);
    b *= 0xc2b2ae35;
    b ^= (b >> 15);
    return b;
}

extern "C" __global__
void hash_table_insert_linear(const long* items, long n, long table_size, 
                               long* keys, long* values, int* success_count) {
    int idx = blockIdx.x * blockDim. x + threadIdx.x;
    if (idx < n) {
        long item = items[idx];
        unsigned int hash = farmhash_32(item);
        long pos = hash & (table_size - 1);
        
        for (long i = 0; i < table_size; ++i) {
            long probe_pos = (pos + i) & (table_size - 1);
            long old_key = atomicCAS((unsigned long long*)&keys[probe_pos], 
                                     (unsigned long long)-1, 
                                     (unsigned long long)item);
            
            if (old_key == -1) {
                atomicAdd((unsigned long long*)&values[probe_pos], 1ULL);
                atomicAdd(success_count, 1);
                break;
            } else if (old_key == item) {
                atomicAdd((unsigned long long*)&values[probe_pos], 1ULL);
                break;
            }
        }
    }
}

extern "C" __global__
void hash_table_insert_quadratic(const long* items, long n, long table_size, 
                                  long* keys, long* values, int* success_count) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        long item = items[idx];
        unsigned int hash = farmhash_32(item);
        long pos = hash & (table_size - 1);
        
        for (long i = 0; i < table_size; ++i) {
            long probe_pos = (pos + i * i) & (table_size - 1);
            long old_key = atomicCAS((unsigned long long*)&keys[probe_pos], 
                                     (unsigned long long)-1, 
                                     (unsigned long long)item);
            
            if (old_key == -1) {
                atomicAdd((unsigned long long*)&values[probe_pos], 1ULL);
                atomicAdd(success_count, 1);
                break;
            } else if (old_key == item) {
                atomicAdd((unsigned long long*)&values[probe_pos], 1ULL);
                break;
            }
        }
    }
}

extern "C" __global__
void hash_table_lookup_linear(const long* queries, long n_queries, long table_size,
                               const long* keys, const long* values, long* results) {
    int idx = blockIdx.x * blockDim. x + threadIdx.x;
    if (idx < n_queries) {
        long query = queries[idx];
        unsigned int hash = farmhash_32(query);
        long pos = hash & (table_size - 1);
        
        for (long i = 0; i < table_size; ++i) {
            long probe_pos = (pos + i) & (table_size - 1);
            long key = keys[probe_pos];
            
            if (key == query) {
                results[idx] = values[probe_pos];
                break;
            } else if (key == -1) {
                results[idx] = 0;
                break;
            }
        }
    }
}

extern "C" __global__
void hash_table_lookup_quadratic(const long* queries, long n_queries, long table_size,
                                  const long* keys, const long* values, long* results) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n_queries) {
        long query = queries[idx];
        unsigned int hash = farmhash_32(query);
        long pos = hash & (table_size - 1);
        
        for (long i = 0; i < table_size; ++i) {
            long probe_pos = (pos + i * i) & (table_size - 1);
            long key = keys[probe_pos];
            
            if (key == query) {
                results[idx] = values[probe_pos];
                break;
            } else if (key == -1) {
                results[idx] = 0;
                break;
            }
        }
    }
}
"""

In [10]:
insert_linear_kernel = cp.RawKernel(HASH_TABLE_KERNEL, 'hash_table_insert_linear')
insert_quadratic_kernel = cp.RawKernel(HASH_TABLE_KERNEL, 'hash_table_insert_quadratic')
lookup_linear_kernel = cp. RawKernel(HASH_TABLE_KERNEL, 'hash_table_lookup_linear')
lookup_quadratic_kernel = cp.RawKernel(HASH_TABLE_KERNEL, 'hash_table_lookup_quadratic')

In [11]:
def next_power_of_2(n):
    return 1 << (n - 1).bit_length()

def run_hash_table(items, load_factor, probe_type, runs=10):
    n_items = len(items)
    
    # Calculate unique count FIRST to size the table correctly
    unique_items = cp.unique(items)
    n_unique = len(unique_items)
    
    # Table size based on UNIQUE items, not total items
    table_size = next_power_of_2(int(n_unique / load_factor))
    
    keys = cp.full(table_size, -1, dtype=cp. int64)
    values = cp.zeros(table_size, dtype=cp.int64)
    success_count = cp.zeros(1, dtype=cp.int32)
    
    insert_kernel = insert_linear_kernel if probe_type == 'linear' else insert_quadratic_kernel
    lookup_kernel = lookup_linear_kernel if probe_type == 'linear' else lookup_quadratic_kernel
    
    blocks = (n_items + 255) // 256
    
    # Warmup
    insert_kernel((blocks,), (256,), (items, n_items, table_size, keys, values, success_count))
    cp.cuda.Stream.null.synchronize()
    
    keys.fill(-1)
    values.fill(0)
    success_count.fill(0)
    
    # Measure insertion time
    insert_times = []
    for _ in range(runs):
        keys.fill(-1)
        values.fill(0)
        success_count. fill(0)
        start = time.time()
        insert_kernel((blocks,), (256,), (items, n_items, table_size, keys, values, success_count))
        cp.cuda.Stream.null.synchronize()
        insert_times.append(time.time() - start)
    
    # Measure lookup time
    n_queries = len(unique_items)
    query_blocks = (n_queries + 255) // 256
    results = cp. zeros(n_queries, dtype=cp.int64)
    
    lookup_kernel((query_blocks,), (256,), (unique_items, n_queries, table_size, keys, values, results))
    cp.cuda.Stream.null.synchronize()
    
    lookup_times = []
    for _ in range(runs):
        results.fill(0)
        start = time.time()
        lookup_kernel((query_blocks,), (256,), (unique_items, n_queries, table_size, keys, values, results))
        cp.cuda.Stream.null.synchronize()
        lookup_times.append(time.time() - start)
    
    actual_load = float(cp.sum(keys != -1)) / table_size
    
    return np.mean(insert_times) * 1000, np.mean(lookup_times) * 1000, actual_load, table_size

In [12]:
print("EXPERIMENT C: HASH TABLE WITH LINEAR PROBING")
print(f"\nDataset: {dataset_size: ,} items  |  Unique items: {true_card: ,}\n")

print(f"{'Load Factor':<15} {'Table Size':<15} {'Actual Load':<15} {'Insertion Time':<18} {'Memory':<12}")
print("-" * 90)

linear_results = []
for lf in [0.10, 0.20, 0.30, 0.40, 0.50, 0.67, 0.75, 0.85, 0.90]: 
    insert_time, lookup_time, actual_load, table_size = run_hash_table(items_array, lf, 'linear')
    linear_results.append((lf, insert_time, lookup_time, actual_load, table_size))
    mem_mb = table_size * 16 / (1024 * 1024)
    print(f"{lf:<15.2f} {table_size:<15,} {actual_load:<15.3f} {insert_time:<12.3f} ms    {mem_mb:<8.1f} MB")

print("\n")

EXPERIMENT C: HASH TABLE WITH LINEAR PROBING

Dataset:  908,173 items  |  Unique items:  16,469

Load Factor     Table Size      Actual Load     Insertion Time     Memory      
------------------------------------------------------------------------------------------
0.10            262,144         0.063           0.100        ms    4.0      MB
0.20            131,072         0.126           0.087        ms    2.0      MB
0.30            65,536          0.251           0.084        ms    1.0      MB
0.40            65,536          0.251           0.083        ms    1.0      MB
0.50            65,536          0.251           0.083        ms    1.0      MB
0.67            32,768          0.503           0.123        ms    0.5      MB
0.75            32,768          0.503           0.123        ms    0.5      MB
0.85            32,768          0.503           0.116        ms    0.5      MB
0.90            32,768          0.503           0.116        ms    0.5      MB




In [13]:
print("EXPERIMENT C: HASH TABLE WITH QUADRATIC PROBING")
print(f"\nDataset: {dataset_size:,} items  |  Unique items: {true_card: ,}\n")

print(f"{'Load Factor':<15} {'Table Size':<15} {'Actual Load':<15} {'Insertion Time':<18} {'Memory':<12}")
print("-" * 90)

quadratic_results = []
for lf in [0.10, 0.20, 0.30, 0.40, 0.50, 0.67, 0.75, 0.85, 0.90]:
    insert_time, lookup_time, actual_load, table_size = run_hash_table(items_array, lf, 'quadratic')
    quadratic_results.append((lf, insert_time, lookup_time, actual_load, table_size))
    mem_mb = table_size * 16 / (1024 * 1024)
    print(f"{lf:<15.2f} {table_size:<15,} {actual_load:<15.3f} {insert_time:<12.3f} ms    {mem_mb:<8.1f} MB")

print("\n")

EXPERIMENT C: HASH TABLE WITH QUADRATIC PROBING

Dataset: 908,173 items  |  Unique items:  16,469

Load Factor     Table Size      Actual Load     Insertion Time     Memory      
------------------------------------------------------------------------------------------
0.10            262,144         0.063           0.097        ms    4.0      MB
0.20            131,072         0.126           0.088        ms    2.0      MB
0.30            65,536          0.251           0.092        ms    1.0      MB
0.40            65,536          0.251           0.085        ms    1.0      MB
0.50            65,536          0.251           0.084        ms    1.0      MB
0.67            32,768          0.503           0.113        ms    0.5      MB
0.75            32,768          0.503           0.113        ms    0.5      MB
0.85            32,768          0.503           0.113        ms    0.5      MB
0.90            32,768          0.503           0.113        ms    0.5      MB




In [14]:
print("COMPLETE PERFORMANCE COMPARISON")
print(f"\n{'Method':<15} {'Load Factor':<15} {'Insertion Time':<18} {'Lookup Time':<18} {'Total Time':<18}")
print("-" * 90)

for lf, ins, lkp, _, _ in linear_results:
    total = ins + lkp
    print(f"{'Linear':<15} {lf:<15.2f} {ins:<12.3f} ms    {lkp:<12.3f} ms    {total:<12.3f} ms")

print()

for lf, ins, lkp, _, _ in quadratic_results:
    total = ins + lkp
    print(f"{'Quadratic': <15} {lf:<15.2f} {ins:<12.3f} ms    {lkp:<12.3f} ms    {total:<12.3f} ms")


COMPLETE PERFORMANCE COMPARISON

Method          Load Factor     Insertion Time     Lookup Time        Total Time        
------------------------------------------------------------------------------------------
Linear          0.10            0.100        ms    0.019        ms    0.119        ms
Linear          0.20            0.087        ms    0.019        ms    0.106        ms
Linear          0.30            0.084        ms    0.019        ms    0.103        ms
Linear          0.40            0.083        ms    0.019        ms    0.102        ms
Linear          0.50            0.083        ms    0.020        ms    0.102        ms
Linear          0.67            0.123        ms    0.022        ms    0.145        ms
Linear          0.75            0.123        ms    0.022        ms    0.144        ms
Linear          0.85            0.116        ms    0.017        ms    0.133        ms
Linear          0.90            0.116        ms    0.017        ms    0.133        ms

Quadratic   

In [13]:
item_hash_table = {}

with open("/home/mithun/Transactional_retail.csv", "r") as f:
    for tid, line in enumerate(f, start=1):
        items = line.strip().split()
        for item in items:
            if item not in item_hash_table:
                item_hash_table[item] = []
            item_hash_table[item].append(tid)

for item, tids in item_hash_table.items():
    print(f"{item}: {tids}")


transactions = {}
for item, tids in item_hash_table.items():
    for tid in tids:
        transactions.setdefault(tid, []).append(item)

# Save it to a PAMI-compatible file
with open("transactions.txt", "w") as f:
    for tid in sorted(transactions):
        f.write("\t".join(transactions[tid]) + "\n")

0: [1, 163, 229, 958, 1395, 1645, 1857, 1859, 2471, 2565, 2981, 3864, 3967, 4084, 4111, 4328, 4334, 4603, 4930, 5012, 5667, 5989, 6292, 6713, 7238, 7466, 7473, 8843, 9034, 9183, 9296, 9392, 10533, 10842, 11148, 11871, 12142, 12149, 12540, 12648, 12656, 12749, 12850, 12971, 13110, 13284, 14054, 14303, 15223, 15561, 16140, 16413, 16700, 16730, 16936, 17317, 17390, 17454, 17531, 17919, 20791, 20874, 21251, 22272, 22712, 24079, 24521, 26410, 28049, 29041, 33557, 38383, 38724, 39065, 39229, 39366, 39498, 39547, 39642, 40796, 41670, 41821, 41927, 42270, 42360, 43109, 43336, 43792, 43810, 44391, 45544, 46321, 46834, 47073, 47150, 47452, 48563, 48584, 48645, 49233, 49254, 49354, 50640, 50732, 51803, 54356, 54358, 54629, 55183, 55205, 55614, 55687, 56772, 57798, 57934, 59201, 59598, 60777, 60998, 61151, 61417, 61557, 62586, 62602, 62738, 62960, 63802, 63813, 65187, 65755, 66051, 66754, 67061, 67226, 67347, 67439, 68205, 71077, 71673, 72044, 72197, 72620, 72899, 72977, 73345, 73381, 73614, 73633

In [ ]:
import PAMI.frequentPattern.basic.Apriori as alg

file  = "transactions.txt"

minSup = 200

apriori = alg.Apriori(file, minSup)

apriori.mine()

patterns = apriori.getPatterns()

print("Apriori:", len(patterns))
print("Runtime (seconds):", apriori.getRuntime())


KeyboardInterrupt: 

In [52]:
import cupy as cp
import time

delimiter = ord('\t')
newline = ord('\n')

In [60]:
start = time.time()

with open(file, 'rb') as f:
    data = f.read()

byte_array = cp.frombuffer(data, dtype=cp.uint8)

# Find lines
line_positions = cp.where(byte_array == newline)[0]
line_positions = cp. concatenate([cp.array([0], dtype=cp.int64), 
                                 line_positions, 
                                 cp. array([byte_array.shape[0]], dtype=cp.int64)])
line_count = int(line_positions.shape[0] - 1)

# Fast count on GPU
count_items_kernel = cp.RawKernel(r"""
extern "C" __global__
void count_items(const unsigned char* data, const long* line_pos,
                 long line_count, unsigned char delim, long* counts) {
    int idx = blockIdx.x * blockDim.x + threadIdx. x;
    if(idx >= line_count) return;
    
    long start = line_pos[idx];
    long end = line_pos[idx + 1];
    long count = 1;
    for(long i = start; i < end; i++) {
        if(data[i] == delim) count++;
    }
    counts[idx] = count;
}
""", 'count_items')

items_per_line = cp.zeros(line_count, dtype=cp.int64)
threads = 256
blocks = (line_count + threads - 1) // threads
count_items_kernel((blocks,), (threads,), 
                   (byte_array, line_positions, cp.int64(line_count), 
                    cp.uint8(delimiter), items_per_line))

# Allocate
index_array = cp.zeros(line_count + 1, dtype=cp.int64)
index_array[1:] = cp.cumsum(items_per_line)
total_items = int(index_array[-1])
items_array = cp.zeros(total_items, dtype=cp. int64)

# Parse AND sort each transaction
parse_kernel = cp.RawKernel(r"""
extern "C" __global__
void parse_and_sort(const unsigned char* data, const long* line_pos,
                    const long* index, long line_count, unsigned char delim,
                    long* items) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if(idx >= line_count) return;
    
    long start = line_pos[idx];
    long end = line_pos[idx + 1];
    long out_idx = index[idx];
    long num_items = 0;
    long val = 0;
    
    // Parse
    for(long i = start; i <= end; i++) {
        unsigned char c = (i < end) ? data[i] : delim;
        if(c == delim) {
            items[out_idx + num_items++] = val;
            val = 0;
        } else if(c >= '0' && c <= '9') {
            val = val * 10 + (c - '0');
        }
    }
    
    // Bubble sort (small transactions, it's fine)
    long n = num_items;
    for(long i = 0; i < n - 1; i++) {
        for(long j = 0; j < n - i - 1; j++) {
            if(items[out_idx + j] > items[out_idx + j + 1]) {
                long temp = items[out_idx + j];
                items[out_idx + j] = items[out_idx + j + 1];
                items[out_idx + j + 1] = temp;
            }
        }
    }
}
""", 'parse_and_sort')

parse_kernel((blocks,), (threads,),
             (byte_array, line_positions, index_array, cp.int64(line_count),
              cp.uint8(delimiter), items_array))

# Fast count with binary search
count_kernel = cp.RawKernel(r"""
extern "C" __global__
void count_itemsets(const long* items, const long* index, long line_count,
                    const long* itemsets, long num_itemsets, long itemset_size,
                    long* counts) {
    int idx = blockIdx.x * blockDim.x + threadIdx. x;
    if(idx >= num_itemsets) return;
    
    long count = 0;
    for(long t = 0; t < line_count; t++) {
        long start = index[t];
        long end = index[t + 1];
        
        bool found_all = true;
        for(long k = 0; k < itemset_size; k++) {
            long item = itemsets[idx * itemset_size + k];
            
            // Binary search
            long left = start, right = end - 1;
            bool found = false;
            while(left <= right) {
                long mid = (left + right) / 2;
                if(items[mid] == item) {
                    found = true;
                    break;
                } else if(items[mid] < item) {
                    left = mid + 1;
                } else {
                    right = mid - 1;
                }
            }
            
            if(! found) {
                found_all = false;
                break;
            }
        }
        if(found_all) count++;
    }
    counts[idx] = count;
}
""", 'count_itemsets')

# Get frequent 1-itemsets
unique_items, item_counts = cp.unique(items_array, return_counts=True)
mask = (unique_items != 0) & (item_counts >= minSup)
current_frequent = unique_items[mask].get().tolist()
frequentPatterns = len(current_frequent)

k = 2
current_frequent = [[x] for x in current_frequent]

# Apriori
while len(current_frequent) >= k:
    # Fast candidate generation
    candidates = []
    for i in range(len(current_frequent)):
        for j in range(i + 1, len(current_frequent)):
            if current_frequent[i][:-1] == current_frequent[j][:-1]:
                candidates.append(current_frequent[i] + [current_frequent[j][-1]])
    
    if not candidates:
        break
    
    # Batch GPU count
    max_size = k
    candidates_flat = cp.array([item for c in candidates for item in c + [0]*(max_size-len(c))], dtype=cp.int64)
    candidates_gpu = candidates_flat.reshape(len(candidates), max_size)
    support_counts = cp.zeros(len(candidates), dtype=cp.int64)
    
    blocks = (len(candidates) + 255) // 256
    count_kernel((blocks,), (256,),
                 (items_array, index_array, cp. int64(line_count),
                  candidates_gpu, cp. int64(len(candidates)), cp.int64(k),
                  support_counts))
    
    # Filter
    support = support_counts.get()
    current_frequent = [candidates[i] for i in range(len(candidates)) if support[i] >= minSup]
    frequentPatterns += len(current_frequent)
    k += 1

end = time.time()
print(f"Time: {end - start:.4f}")
print(f"Frequent Patterns: {frequentPatterns}")

Time: 1.8696
Frequent Patterns: 2190


In [ ]:
minSup = 300